In [1]:
import os
import pandas as pd  # to load data into DataFrame
import numpy as np   # to deal with arrays and numerical values
import datetime #from datetime import date
import traceback
from ctREFPROP.ctREFPROP import REFPROPFunctionLibrary

import sqlite3

from datetime import datetime, timedelta # to get date and time # to represent the difference between two dates or times

import warnings
warnings.filterwarnings(action='ignore')


In [2]:
# Connect to the Db
connection = sqlite3.connect('C:/KHU_DRL/NED_SQLite.db')

sql = "SELECT * FROM odu2_data ORDER BY timestamp DESC LIMIT 4320 " #3days / 1min period data
odu1_data = pd.read_sql_query(sql, connection)

connection.close()

# 실외온도만 가져옴 / 건물 파악은 이번 단계에서는 실시하지 않음
# 동계에는 건물체에 대한 파악도 1달에 한 번씩은 시행할 것

In [52]:
odu1_data['ambient'] = odu1_data['ODU_outtemp']
# odu1_data = odu1_data.set_index('timestamp')

In [53]:
odu_ambient_df = odu1_data[['timestamp', 'ambient']]

In [54]:
odu_ambient_df['timestamp'] = pd.to_datetime(odu_ambient_df.timestamp)
odu_ambient_df = odu_ambient_df.set_index('timestamp')

In [55]:
odu_ambient_df_avg = odu_ambient_df.groupby([pd.Grouper(freq = '5min')]).mean()

In [56]:
odu_ambient_df_avg = odu_ambient_df_avg.reset_index()

In [57]:
odu_ambient_df_avg['hour'] = odu_ambient_df_avg['timestamp'].dt.hour
odu_ambient_df_avg['min'] = odu_ambient_df_avg['timestamp'].dt.minute

In [58]:
odu_ambient_df_avg.drop(labels = ['timestamp'],axis=1, inplace=True)

In [59]:
df_period = odu_ambient_df_avg.groupby(['hour', 'min'], as_index=False).mean()
# 최근 3일간의 시간대별 외기온도 변화를 5분 단위로 표시함(평균값)

In [60]:
# df_june.to_csv('test.csv')

In [61]:
# 고압 포화 온도측 ΔT 예측 / 공통 함수
def condensing_temp_delta_t(target_press, op_rate, id_temp):
    root = 'C:/Refprop_dll' # Change this path as needed
    RP = REFPROPFunctionLibrary(root)
    RP.SETPATHdll(root)
    
    MKS = RP.GETENUMdll(0, "MKS").iEnum
    
    fluid = 'R410A.MIX'
    FLDpath1 = os.path.join(root,"mixtures",fluid)
    sm = RP.SETMIXdll(fluid,'HMX.BNC','IIR') # in case of mixture, this sentence should be defined
    RP.SETREFdll('IIR',2,sm.z,0,0,0,0)
    z = sm.z  
        
    #r = RP.REFPROPdll(fluid,"PS","H",MKS,0,0,cond_press,entropy,z)
    r = RP.REFPROPdll(fluid,"PQ","T",MKS,0,0,(target_press*1.03)*98, 1.0 ,z)
    eva_temp = r.Output[0] 
    eva_temp = eva_temp-273.15      # 섭씨 온도
    
    delta_eva_temp = id_temp - eva_temp
    
    delta_cond_temp = (9-target_press)*1.5 + (op_rate*5) + (delta_eva_temp-12)*0.5
    
    return delta_cond_temp

In [62]:
# 고압 예측 / 공통 함수
def cond_press(od_temp, delta_cond_temp):
    root = 'C:/Refprop_dll' # Change this path as needed
    RP = REFPROPFunctionLibrary(root)
    RP.SETPATHdll(root)
    
    MKS = RP.GETENUMdll(0, "MKS").iEnum
    
    fluid = 'R410A.MIX'
    FLDpath1 = os.path.join(root,"mixtures",fluid)
    sm = RP.SETMIXdll(fluid,'HMX.BNC','IIR') # in case of mixture, this sentence should be defined
    RP.SETREFdll('IIR',2,sm.z,0,0,0,0)
    z = sm.z  
    
    #r = RP.REFPROPdll(fluid,"PS","H",MKS,0,0,cond_press,entropy,z)
    r = RP.REFPROPdll(fluid,"TQ","P",MKS,0,0, od_temp+delta_cond_temp+273.15, 1.0 ,z)
    cond_press = r.Output[0] 
    cond_press = cond_press/98-1.03      # 게이지 압력    
    
    if cond_press < 25:
        cond_press = 25
    else:
        pass
    
    return cond_press

In [63]:
# 압축기 일 계산 / 공통 함수
def comp_work(target_press, cond_press):
    root = 'C:/Refprop_dll' # Change this path as needed
    RP = REFPROPFunctionLibrary(root)
    RP.SETPATHdll(root)
    
    MKS = RP.GETENUMdll(0, "MKS").iEnum
    
    fluid = 'R410A.MIX'
    FLDpath1 = os.path.join(root,"mixtures",fluid)
    sm = RP.SETMIXdll(fluid,'HMX.BNC','IIR') # in case of mixture, this sentence should be defined
    RP.SETREFdll('IIR',2,sm.z,0,0,0,0)
    z = sm.z  
    
    r = RP.REFPROPdll(fluid,"PQ","T",MKS,0,0, (target_press+1.03)*98, 1.0 ,z)
    suction_temp = r.Output[0] #K
    #print(suction_temp)
    
    r = RP.REFPROPdll(fluid,"PT","H;S",MKS,0,0, (target_press+1.03)*98, suction_temp+5 ,z)
    enthalpy_1 = r.Output[0] #kJ/kg
    entropy_1 = r.Output[1]
    #print(enthalpy_1)
    #print(entropy_1)
    
    r = RP.REFPROPdll(fluid,"PS","H",MKS,0,0, (cond_press+1.03)*98, entropy_1 ,z)
    enthalpy_2 = r.Output[0] #KkJ/kg    
    #print(enthalpy_2)    
    
    comp_work = enthalpy_2 - enthalpy_1
    
    return comp_work


In [64]:
def oper_status(hour_1, min_1): 
    #운전 시간을 일과 시간으로 한정하고 정책을 찾아냄
    #고도화되면 운전 시간마저 학습 대상으로 해야 함
    #8:30부터 예열 운전하는 것으로 하였음(30분 전부터)
    if (hour_1 == 8 and min_1 >= 30) or (hour_1 > 8 and hour_1 < 18):
        temp = 1
    else:
        temp = 0
        
    return temp

In [65]:
def inner_heat_status(hour_1):
    #일과 시간을 한정지었음 / 근무 시간도 학습 대상이 될 수 있음
    if hour_1 >= 9 and hour_1 < 18:
        temp = 1
    else:
        temp = 0
        
    return temp

In [66]:
scenario_ambient_temp = [-0.3, -0.1, 0.1] #[-0.3, -0.17, -0.03, 0.1] 230912 isko offset을 넣기 위해 -0.17과 -0.03을 빼고 -0.1을 넣음
scenario_delta_temp = [-0.3, -0.15, -0.0] # [-0.3, -0.15, 0.0] [-0.3, -0.2, -0.1, 0.0]
scenario_offset = [0, 0.2] # 230912 isko 

In [67]:
def target_press_by_scenario(coef_amb, code_delta, offset, ambient_temp, set_temp, inner_temp, op_status): #isko 230912 offset 추가
    target = coef_amb*(ambient_temp-30) + code_delta*(inner_temp-set_temp) + offset + 8 #35 -> 30, isko 230912 offset 추가
    target = max(min(target, 9.5), 6.5)
    if op_status==0:
        target = 0
    else:
        pass
    
    return round(target,1)

In [68]:
# odu1(3층) 소비전력 예측
def odu1_power_input(comp_work, op_rate):
    odu_power_input = -3220 + 3305.5*op_rate + 138.72*comp_work
    
    return odu_power_input

In [69]:
# odu2(2층) 소비전력 예측
def odu2_power_input(comp_work, op_rate):
    odu_power_input = -2420 + 5795*op_rate + 109.7*comp_work
    
    return odu_power_input

In [70]:
Ri  =       0.172560  
Ro  =       1.798633 
Ci  =    1157.592162  
Ce  =  100446.997227 

Atp =   8.769830e-02
Aac =   -3.500000e+01 
Across=  2.642250e+00 
Aih   =  2.232786e-10
C_0   =  2.227854e+00 
C_1   =  2.023524e+00 

# Ri   =     0.296298 
# Ro   =     1.262322 
# Ci   =  1263.760214 
# Ce   = 81085.675231 

# Atp  =    0.216114 
# Aac  =  -28.649686 
# Across =  2.307840 
# Aih  =    0.390904 
# C_0  =   -0.585024 
# C_1  =    2.251970  

In [71]:
def odu1_ti_t(ti_t_1, te_t_1, target_press, op_rate, inner_heat_status):
 
    dt = 5*60 #초
    
    # ti_t = ti_t_1 + dt / Ci * ( (te_t_1 - ti_t_1)/Ri + Atp*(target_press - C_sp) + Aac*op_status + Aih*inner_heat_status + C_0 )

    if op_rate == 0 :
        ti_t = ti_t_1 + dt / Ci * ( (te_t_1 - ti_t_1)/Ri + Aih*inner_heat_status + C_1 )
        # ti[t] = ti[t-1] + dt / Ci * ( (te[t-1]-ti[t-1])/Ri + Aih*inner_heat[t-1] + C_1 )
        # 외부에 드나드는 열량이 있다고 보는 것이 타당하지 않겠는가?
        # 에어컨을 켜지 않았다면 환기를 시도했을 가능성도 있다
    else:
        ti_t = ti_t_1 + dt / Ci * ( (te_t_1 - ti_t_1)/Ri + Atp*(target_press) + Aac*op_rate + Across*target_press*op_rate + Aih*inner_heat_status + C_0 )
        # ti[t] = ti[t-1] + dt / Ci * ( (te[t-1]-ti[t-1])/Ri + Atp*(phi_tp[t-1]) + Aac*phi_ac[t-1] + Across*(phi_tp[t-1])*phi_ac[t-1] + Aih*inner_heat[t-1] + C_0 )    
    
    return ti_t

In [72]:
def odu1_te_t(ti_t_1, te_t_1, to_t_1):#, target_press, op_status, inner_heat_status

    dt = 5*60 #초
    
    te_t = te_t_1 + dt / Ce * ( (ti_t_1 - te_t_1)/Ri + (to_t_1 - te_t_1)/Ro )
    
    return te_t

In [73]:
sum_power = 0
cnt_i = 0


df_score = pd.DataFrame([], columns = ['coef_amb', 'coef_delta_t', 'coef_offset', 's_comfort', 's_energy', 'total_score']) #isko 230912

In [74]:
# month_table = ['June', 'July', 'August'] #축열 테스트

In [75]:
for i_month in range(0,1): # 축열 테스트 (0,3)
#     df_june = pd.read_csv(rawdata_dir + month_table[i_month] +'.csv')
#     df_june = df_june.dropna(axis=0)
#     df_june['min']=0
#     df_5min = df_june.copy()

#     for i in range(1,25): #24

#         for j in range(1,12): #12
#             if i == 24:
#                 new_temp = ( df_june.loc[i-1, 'ambient']*(12-j) + df_june.loc[0, 'ambient']*j ) / 12
#             else:
#                 new_temp = ( df_june.loc[i-1, 'ambient']*(12-j) + df_june.loc[i, 'ambient']*j ) / 12
#             df_new = pd.DataFrame([[df_june.loc[i-1, 'hour'], new_temp, j*5]], columns=df_june.columns)
#             df_5min = pd.concat([df_5min.iloc[:12*(i-1)+j], df_new, df_5min.iloc[12*(i-1)+j:]], ignore_index = True)
            
#             df_period = df_5min
        
    for ii in range(0,3): # 축열 테스트 (0,4): 230912 isko
        for jj in range(0,3): # 축열 테스트 (0,3)
            for kk in range(0,2): # 230912 isko offset 점검 추가
                cnt_s = 0
                scors_s = 0
                sum_power = 0
                pre_totalpower = 0
                df_result = pd.DataFrame([], columns = ['day', 'hour', 'min', 't_i', 't_o', 't_wall', 'target_press', 'op_rate', 'sum_power'])

                for i in range(0,5): #5일 고려
                    for j in range(0,288): #288, 24시간 고려, 5분 간격
                        coef_amb = scenario_ambient_temp[ii]
                        code_delta = scenario_delta_temp[jj]

                        set_temp = 24.0

                        to = df_period.loc[j, 'ambient']

                        if i==0 and j==0:
                            ti_0 = 25.0
                            to_0 = df_period.loc[0, 'ambient']
                            te_0 = df_period['ambient'].mean() # max -> mean, 30 #( Ri*to_0 + Ro*ti_0 ) / (Ri + Ro)

                            ti_t_1 = ti_0
                            te_t_1 = te_0
                            to_t_1 = to_0

                        else:            
                            to_t = df_period.loc[j, 'ambient']

                            if j == 0 :
                                pass
                            else:     
                                hour_temp = df_period.loc[j-1, 'hour']
                                min_temp = df_period.loc[j-1, 'min']

                            op_st_1 = oper_status(hour_temp, min_temp)
                            inner_st_1 = inner_heat_status(hour_temp)
                            occupancy_status = inner_st_1 #일단 근무 시간과 발열 시간을 동일하게 처리한다

                            if op_st_1 == 1.0:
                                if ti_t_1 <= set_temp - 1.0:
                                    op_st_1 = 0.0
                                else:
                                    pass
                            else:
                                pass
                            op_st_1 = op_st_1 * 0.75  #28.8.4 표준 운전율을 75%로 하고자 함

                            target_press = target_press_by_scenario(scenario_ambient_temp[ii], scenario_delta_temp[jj], scenario_offset[kk], to_t_1, set_temp, ti_t_1, op_st_1) #isko 230912
                            #기존 제어 =============================================
                            # if target_press != 0:
                            #     target_press = 8
                            # else:
                            #     pass
                            #축열 테스트 =============================================

                            #축열 테스트 =============================================
                            # if target_press != 0:
                            #     if hour_temp < 11: #9~11시 (2시간)
                            #         target_press = target_press - 0.2
                            #     elif hour_temp > 15: #16~18시 (2시간)
                            #         target_press = target_press + 0.3
                            #     else:
                            #         pass
                            # else:
                            #     pass
                            #축열 테스트 =============================================

                            ti_t = odu1_ti_t(ti_t_1, te_t_1, target_press, op_st_1, inner_st_1)

                            te_t = odu1_te_t(ti_t_1, te_t_1, to_t_1)

                            if occupancy_status == 1:
                                cnt_s = cnt_s + 1
                                if ti_t - set_temp < 1:
                                    scors_s = scors_s + 100
                                elif ti_t - set_temp < 2:
                                    scors_s = scors_s + 90
                                elif ti_t - set_temp < 3:
                                    scors_s = scors_s + 70
                                elif ti_t - set_temp < 4:
                                    scors_s = scors_s + 50
                                else:
                                    scors_s = scors_s + 0

                            totalpower = 0
                            if op_st_1 > 0: #운전율 0.75
                                cond_delta_t = condensing_temp_delta_t(target_press, 0.75, ti_t_1)
                                cond_p = cond_press(to_t_1, cond_delta_t)
                                comp_power = comp_work(target_press, cond_p)
                                totalpower = odu2_power_input(comp_power, 0.75) #odu1 → odu2
                                #print(totalpower)
                                if pre_totalpower == 0:
                                    totalpower = totalpower * 1.5
                                else:
                                    pass
                                sum_power = sum_power + totalpower #(W*5min)
                                cnt_i = cnt_i + 1

                            # if  df_period.loc[j, 'hour'] >=8 and df_period.loc[j, 'hour'] <= 18:
                            # print(i+1, '   ', int(df_period.loc[j, 'hour']), ':', df_period.loc[j, 'min'], '   ', round(ti_t,1),'   ', round(to_t,1),'   ', round(te_t,2), '   ', target_press, '   ', op_st_1)

                            # df_result = df_result.append([[i+1, int(df_period.loc[j, 'hour']),  df_period.loc[j, 'min'],  round(ti_t,1), round(to_t,1), round(te_t,2), target_press, op_st_1, sum_power]], 
                            #                              columns = ['day', 'hour', 'min', 't_i', 't_o', 't_wall', 'target_press', 'op_rate', 'sum_power'], ignore_index=True)
                            df_new_line = ({'day':i+1, 'hour':int(df_period.loc[j, 'hour']), 'min':df_period.loc[j, 'min'], 't_i':round(ti_t,1), 't_o':round(to_t,1), 
                                            't_wall':round(te_t,2), 'target_press':target_press, 'op_rate':op_st_1, 'sum_power':sum_power})
                            df_result = df_result.append(df_new_line, ignore_index=True)

                            ti_t_1 = ti_t
                            te_t_1 = te_t
                            to_t_1 = to_t

                            pre_totalpower = totalpower

                score_comfort = scors_s / cnt_s

                power_std = 5 * 8 * 8 * 2.9 / 3 * 0.75 #5일, 8시간, 8HP, 2.9kW, 표준 효율 3.0, 표준 운전율 75% [kWh
                sum_power_1 = sum_power/12/1000 #[kWh]
                #score_energy = (1-sum_power_1/power_std)*100 + 70
                score_energy = (sum_power_1/power_std)*(-100)

                score_total = score_comfort + score_energy

                df_new_result = ({'coef_amb':scenario_ambient_temp[ii], 'coef_delta_t':scenario_delta_temp[jj], 
                                  'coef_offset':scenario_offset[kk], 's_comfort':score_comfort, 's_energy':score_energy, 'total_score':score_total}) #isko 230912
                df_score = df_score.append(df_new_result, ignore_index=True)

                df_result.to_csv('C:/KHU_DRL/output/ODU2_' + datetime.today().strftime('%Y%m%d_%H%M_') + str(ii) + '_' + str(jj) + '_' + str(kk) + '.csv') #isko 230912
                print(datetime.today().strftime('%Y%m%d_%H%M_') + str(ii) + '_' + str(jj) + '_' + str(kk)) #isko 230912

20230912_1054_0_0_0
20230912_1055_0_0_1
20230912_1056_0_1_0
20230912_1057_0_1_1
20230912_1057_0_2_0
20230912_1058_0_2_1
20230912_1059_1_0_0
20230912_1100_1_0_1
20230912_1101_1_1_0
20230912_1102_1_1_1
20230912_1102_1_2_0
20230912_1103_1_2_1
20230912_1104_2_0_0
20230912_1105_2_0_1
20230912_1105_2_1_0
20230912_1106_2_1_1
20230912_1107_2_2_0
20230912_1108_2_2_1


In [76]:
df_score
df_score.to_csv('C:/KHU_DRL/2nd_score_' + datetime.today().strftime('%Y%m%d_%H%M.csv'))

In [77]:
best_policy = df_score['total_score'].idxmax()

df_result = pd.DataFrame([[df_score.loc[best_policy, 'coef_amb'], df_score.loc[best_policy, 'coef_delta_t'], df_score.loc[best_policy, 'coef_offset'] ]]) #isko 230912

In [78]:
df_result.to_csv('C:/KHU_DRL/2nd_policy.csv', header=None, index=False)

In [79]:
datetime.today().strftime('%Y%m%d_%H%M')

'20230912_1108'

In [84]:
fv = open('C:/KHU_DRL/ODU2_policy_history.txt', 'a')

In [85]:

fv.write(datetime.today().strftime('%Y%m%d_%H%M') + ' / ' + 
         str(df_score.loc[best_policy, 'coef_amb']) + ' / ' + 
         str(df_score.loc[best_policy, 'coef_delta_t']) + ' / ' + 
         str(df_score.loc[best_policy, 'coef_offset']) + '\n') #isko 230912

34

In [86]:
fv.close()